# Uebung 7: Data Cleaning - Airbnb Berlin

**DataPy - WiSe25/26 - Woche 7**

## Lernziele

- Datensaetze systematisch inspizieren
- Datenqualitaetsprobleme identifizieren
- Data-Cleaning-Pipeline entwickeln
- Bereinigte Datensaetze speichern

## Datensatz
Inside Airbnb - Berlin Listings

## Teil 1: Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

pd.set_option('display.max_columns', None)
print("Bibliotheken importiert")

In [ ]:
df = pd.read_csv('listings.csv')
df_clean = df.copy()
print(f"Geladen: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")

## Teil 2: Inspektion

In [ ]:
important = ['id', 'name', 'neighbourhood_cleansed', 'property_type',
             'room_type', 'accommodates', 'bedrooms', 'price',
             'number_of_reviews', 'review_scores_rating']

print("Shape:", df.shape)
print("\nWichtige Spalten:")
display(df[important].head())
print("\nPreis-Beispiele:")
print(df['price'].head(5))

## Teil 3: Fehlende Werte

In [ ]:
missing = df_clean[important].isnull().sum()
missing_pct = (missing / len(df_clean)) * 100
missing_df = pd.DataFrame({
    'Spalte': missing.index,
    'Fehlend': missing.values,
    'Prozent': missing_pct.values
})
missing_df = missing_df[missing_df['Fehlend'] > 0]
display(missing_df)

## Teil 4: Preis-Konvertierung

In [ ]:
def clean_price(price_str): #Preis Strings in Zahl umwandeln
    if pd.isna(price_str): #falls Wert fehlt (NaN)
        return np.nan #NaN zurückgeben
    #String erzwingen,  $-Zeichen und Tausendertrennzeichen entfernen
    cleaned = str(price_str).replace('$', '').replace(',', '')
    try:
        return float(cleaned)
    except:
        return np.nan

df_clean['price_num'] = df_clean['price'].apply(clean_price)
print("Preis konvertiert")
print(df_clean['price_num'].describe())

## Teil 5: Duplikate

In [ ]:
dups = df_clean.duplicated(subset=['id']).sum()
print(f"Duplikate: {dups}")
df_clean.drop_duplicates(subset=['id'], inplace=True)
print(f"Nach Entfernung: {df_clean.shape[0]} Zeilen")

## Teil 6: Ausreisser

In [ ]:
print("Preis-Statistiken:")
print(df_clean['price_num'].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot(df_clean['price_num'].dropna())
axes[0].set_title('Alle Preise')

filtered = df_clean[df_clean['price_num'] < 500]
axes[1].boxplot(filtered['price_num'].dropna())
axes[1].set_title('Preise < 500')
plt.show()

## Teil 7: Feature Engineering

In [ ]:
df_clean['price_cat'] = pd.cut(df_clean['price_num'],
                                bins=[0, 50, 100, 200, 10000],
                                labels=['guenstig', 'mittel', 'teuer', 'luxus'])

df_clean['has_reviews'] = df_clean['number_of_reviews'] > 0
print("Features erstellt")

## Teil 8: Validierung

In [ ]:
print("VERGLEICH")
print(f"Original: {df.shape}")
print(f"Bereinigt: {df_clean.shape}")
retention = (df_clean.shape[0]/df.shape[0])*100
print(f"Retention: {retention:.1f}%")

## Teil 9: Speichern

In [ ]:
export_cols = ['id', 'name', 'neighbourhood_cleansed', 'latitude',
               'longitude', 'property_type', 'room_type', 'accommodates',
               'bedrooms', 'beds', 'price_num', 'number_of_reviews',
               'review_scores_rating', 'price_cat', 'has_reviews']

export_cols = [c for c in export_cols if c in df_clean.columns]

output = 'airbnb_berlin_cleaned.csv'
df_clean[export_cols].to_csv(output, index=False)
print(f"Gespeichert: {output}")
print(f"Zeilen: {df_clean.shape[0]}")
print(f"Spalten: {len(export_cols)}")

## Reflexion

1. Groesste Herausforderungen?
2. Schwierigste Entscheidungen?
3. Erkenntnisse ueber Airbnb Berlin?
4. Zuversicht (1-10)?
5. Fragen fuer Woche 8?